# AIショッピングアシスタント向け評価の構築

このハンズオンでは、AIエージェントの**評価を構築**し、その評価を用いて問題を体系的に特定・修正する方法を学びます。

`boutique` は、商品の価格を検索し、計算を行うシンプルなショッピングアシスタントです。おおむね動作しますが、完全ではありません。どこで、なぜ失敗するのかを正確に特定し、修正することが本演習の課題です。

---

**Partner Basecamp参加者向けの資料です。** 研修資料としての複製・再配布はご遠慮ください。ここで扱うパターンは、皆さまのお客様案件で自由にご活用いただけます。

In [ ]:
# このカーネルに依存パッケージをインストール。再実行しても安全で、制限付きのPython（PEP 668）でも動作する
import importlib.util, os, subprocess, sys

# ── 環境ガード ────────────────────────────────────────────────────────────
# 下のインストール処理と同じセルに置き、スキップできないようにしている。これにより、
# 素のシステムPythonにパッケージがインストールされることはない（従来の最悪のケースは、
# IT部門が管理するマシンでの --break-system-packages の実行）。ここで停止した場合は、
# SETUP.mdの「ノートブックが停止する理由」を参照。
def _in_isolated_env():
    """実行中のカーネルがvenv／virtualenv／conda環境、またはColabであればTrueを返す。
    判定はインタープリター自身（sys.*）に基づく。起動元のシェルから引き継いだ
    有効化用の環境変数は、sys.executableが実際にその環境の内部にある場合に
    限り信頼する。有効化済みのターミナルから起動したシステムPythonの
    カーネルもVIRTUAL_ENVを引き継ぐが、この場合は
    判定を通してはならない。"""
    if "google.colab" in sys.modules:
        return True  # Colabは使い捨ての独自ランタイムでサンドボックス化されている
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405のvenv（python -m venv）。大半のconda環境も該当
    if hasattr(sys, "real_prefix"):
        return True  # 従来のvirtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # シェルから引き継いだ情報にすぎず、カーネルは別の場所にある
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # condaの共有 `base` は分離環境とみなさない
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  環境ガードを回避しました（BASECAMP_ALLOW_SYSTEM_PYTHON=1）。"
          "意図的にこのPythonへインストールします。")
elif not _in_isolated_env():
    _HEAD = "✗ システムPythonを検出しました。何もインストールせずに停止しました"
    _BODY = (
        "ここにパッケージをインストールすると、マシン全体のPythonが変更されます。\n"
        "会社が管理するノートPCでは、IT部門への申請が必要になる場合があります。\n"
        "\n"
        "対処方法（2ステップ）：\n"
        "  1. ターミナルで、リポジトリのルートから次を実行してください：\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windows: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. VS Codeで、右上のカーネル名をクリック → Select Another Kernel →\n"
        "     Python Environments → 末尾が.venvの環境を選択 → このセルを再実行してください。\n"
        "\n"
        "condaをご利用の場合は、`conda activate <env>`（`base` 以外）を実行してから、そのカーネルを選択してください。\n"
        "ご自身で管理するマシンを使うファシリテーターは、BASECAMP_ALLOW_SYSTEM_PYTHON=1 で回避できます。"
    )
    _shown = False
    try:  # 下のAPIキー確認と同じバナー表示（緑のボックス、エラー時は赤）
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "環境ガードによりこのセルを停止しました。上のメッセージをご確認ください。"
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ 環境ガード：分離されたインタープリターを検出しました。安全にインストールできます")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements: (import_name, pip_spec) のリスト。不足しているものだけを、
    実行中のインタープリターにインストールする。通常のインストール、ユーザー領域、
    PEP 668の上書き（ユーザー領域を優先し、システム全体は最後の手段）の順に試行する。
    いずれの試行も出力を表示しない（pipの出力はストリーム表示せずに取り込む）。そのため、
    制限付きのPython（HomebrewやDebian、PEP 668）でフォールバックが実際に成功した場合に、
    'externally-managed-environment' の長いエラー文が表示されることはない。
    すべての方法が失敗した場合に限り、生のトレースバックではなく、
    venvによる対処方法とともに原因を表示する。"""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + "（初回のみ）。しばらくお待ちください…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "（pipからの出力なし）"
    raise SystemExit(
        "\n  インストールできませんでした：" + ", ".join(missing) + "\n"
        "  このPythonは制限付き（PEP 668）か、オフラインの状態です。最も手早い対処方法はvenvの利用です：\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windowsの場合はSETUP.mdを参照\n"
        f"      pip install {' '.join(missing)}\n"
        "  その後、VS Code（右上のカーネル選択）で.venvのインタープリターを選択し、Run Allを実行してください。\n"
        "  社内プロキシやPyPIのブロックが原因の場合は、リポジトリのルートにあるSETUP.mdをご参照ください。\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ 依存パッケージの準備が完了しました")

### セットアップ：Claudeへの接続

最初に次のセルを実行してください。セットアップセルは、初回実行時に **`.env` ファイル**を作成します（gitignoreの対象のため、キーがコミットされることはありません）。ファイルを開き、`ANTHROPIC_API_KEY=` の後ろにキーを貼り付けて保存し、セルを再実行してください。キーはカーネルを再起動しても保持されるため、貼り付けは1回で済みます。*（`.env` がまだない場合は、代わりに、入力した文字が表示されない入力欄でキーの入力を求められます。）* 緑色の **「✓ APIキーを確認しました」** バナーが表示されれば、接続は完了です。赤いバナーが表示された場合は、表示された指示に従ってセルを再実行してください。

In [ ]:
import os

def _status(ok, msg):
    """ノートブックでは緑／赤のバナー、スクリプトとして実行した場合はプレーンテキストで表示する。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("ノートブックのカーネル外のため、プレーンテキストのバナーを使用する")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Claudeへの接続：Anthropic APIまたはAmazon Bedrock ──
# どちらの認証情報でも動作する。種類はこのセルが自動で判別する
#   Anthropic API : ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock: AWS_BEARER_TOKEN_BEDROCK=...  plus  AWS_REGION=us-east-1
# 使用する認証情報を、このセルが作成する.envファイル（gitignoreの対象のため、コミットされない）に記入するか、
# シェルでexportする。シェルの値は.envファイルの値より優先される。
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """作業ディレクトリから上位へたどり、最も近い既存の.envを探す（ルートに.envを1つ置けば
    すべての演習で共用できる）。まだ存在しない場合はリポジトリのルートを指し、
    ノートブックを単独で開いた場合はこのフォルダーを指す。"""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# 簡易的な.envパーサー（python-dotenvに依存しない）。実行のたびに読み直すため、キーを貼り付けて
# 再実行すれば反映される。環境（シェル／Claude Code／CI）に実際の値がある場合は
# そちらが優先され、プレースホルダーが残ることはない。
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """黄色の警告バナーを表示して停止する。数セル先ではなく、この時点でセットアップを失敗させるため。"""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("認証情報が設定されていません。上のメッセージをご確認ください。")


if PROVIDER is None:
    _needs_credentials(
        "📋 続行するには認証情報を追加してください",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Anthropic APIをご利用の場合は、次を設定してください：\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Amazon Bedrockをご利用の場合は、次の両方を設定してください：\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # モデルを有効化しているリージョン\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrockにはリージョンの指定が必要です",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "Bedrockのモデルを有効化しているリージョンを追記してください。例：\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )


def _model(name):
    """BedrockのモデルIDには `anthropic.` というプレフィックスが付きます。Anthropic APIでは、プレフィックスなしのIDを使用します。"""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# この演習で使用するモデル。利用中のプロバイダーに合わせてIDを解決する
MODEL = _model("claude-sonnet-5")        # この演習の主力モデル
FAST_MODEL = _model("claude-haiku-4-5")  # 低コストかつ高速（接続確認、ジャッジ用）
BIG_MODEL = _model("claude-opus-4-8")    # より大きなモデルを試す場合


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# 接続確認：認証情報に加え、このモデルを実際に利用できるかも検証する
# Bedrockでは、キーが有効でも、アカウント／リージョンでモデルが有効化されていなければ404になることがある。
# そのため、認証情報の形式を確認するだけでなく、実際のモデルIDにpingを送る。
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("モデルを利用できません。上のメッセージをご確認ください。")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "Bedrockキーが拒否されました。AWS_BEARER_TOKEN_BEDROCKの値と、"
                       "Bedrockの呼び出し権限が付与されていることを確認してから、このセルを再実行してください。")
    else:
        _status(False, "キーが拒否されました。このセルを再実行し、キー全体を貼り付けてください"
                       "（sk-ant-で始まります）。")
    raise SystemExit("認証情報が受け付けられませんでした。このセルを再実行して、もう一度お試しください。")
except Exception as exc:
    _status(False, "APIに接続できませんでした（" + type(exc).__name__ + "）。"
                   "ネットワーク接続を確認してから、このセルを再実行してください。")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # 以降のセルや !python がこの値を参照する
        _status(True, "APIキーを確認しました。Claudeに接続されています。")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# 実際に使用するクライアント。非ストリーミング呼び出しで max_tokens>21333 を使うには、長めのタイムアウトが必要
client = _make_client(timeout=900.0)


## 評価が必要な理由

Claudeを活用した開発では、「何回か試して動けばよい」という進め方では不十分です。感覚的な確認も重要ですが、評価によって次の3点が得られます。

- **ベースライン**：現時点でのエージェントの性能を把握できます。どの種類のクエリで、どのような結果になるかが明確になります。
- **フィードバックループ**：プロンプト、ツール設計、モデルのいずれかを変更して再実行し、実際に改善したか、リグレッションが生じていないかを確認できます。
- **判断の根拠**：お客様へのリリース前に、感覚だけでなく数値で品質を示せます。

本日のハンズオンの流れ：

1. **エージェントの把握**：何が動作し、何が動作しないかを確認する
2. **評価タスクの定義**：エージェントの機能とエッジケースを網羅する
3. **評価の実行**：結果を体系的に確認する
4. **エージェントの改善**：評価結果に基づいて改善し、再実行で検証する
5. **LLM-as-Judge採点者（grader）の追加**：単純な文字列一致では確認できないクエリに対応する

---

## パート1：エージェントの把握

`boutique` は、ツールを2つ備えたシンプルな単一ターンのエージェントです。

| ツール | 機能 |
|------|-------------|
| `get_product(product)` | カタログから商品の価格を返す |
| `calculate(op, input1, input2)` | 基本的な計算処理 |

エージェントループはシンプルです。

```
User query → Claude decides what to do → Tool call (if needed) → Tool result → ... → Final answer
```

下のコードにひととおり目を通してください。次の箇所は、後で改めて確認することがあります。
- **ツール仕様**（Claudeが各ツールについて把握している情報）
- **システムプロンプト**
- **ツールの実装**（ツールが呼び出されたときに実際に実行される処理）

In [ ]:
import math
from anthropic import Anthropic
from anthropic.types import ToolUseBlock, TextBlock

# ── 設定 ──────────────────────────────────────────────────────────────

MODEL = FAST_MODEL  # Haiku。セットアップで利用中のプロバイダーに合わせて解決済み（エイリアスのため、旧モデルの廃止時も404にならない）
SYSTEM_PROMPT = "You are a helpful assistant."

# ハーネス用クライアント：SDKのデフォルト設定（長いタイムアウト、標準のリトライ）。上の確認用
# クライアントは30秒のタイムアウト／1回のリトライと短く、1トークンのpingには適しているが、
# 評価全体の実行には不安定すぎる。そのため作り直す（キーはすでに環境変数にある）。
client = Anthropic()

# ── ツールの実装 ──────────────────────────────────────────────────────

def get_product(product: str):
    catalog = {
        "jeans": 49.99,
        "shirt": 29.99,
        "dress": 59.99,
        "jacket": 89.99,
        "sneakers": 74.99,
        "hat": 19.99,
        "socks": 9.99,
        "hoodie": 44.99,
        "shorts": 34.99,
        "t-shirt": 24.99,
        "sweater": 54.99,
        "belt": 24.99,
    }
    return catalog[product]


def calculate(op: str, input1: float, input2: float):
    if op == "+": return input1 + input2
    elif op == "-": return input1 - input2
    elif op == "*": return input1 * input2
    elif op == "/": return input1 / input2
    elif op == "**": return input1 ** input2

TOOL_REGISTRY = {
    "get_product": get_product,
    "calculate": calculate,
}

# ── ツール仕様（Claudeに送信）────────────────────────────────────────

GET_PRODUCT_SPEC = {
    "name": "get_product",
    "description": "get_product",
    "input_schema": {
        "type": "object",
        "properties": {
            "product": {
                "type": "string",
                "description": "product",
            },
        },
        "required": ["product"],
    },
}

CALCULATE_SPEC = {
    "name": "calculate",
    "description": "calculator",
    "input_schema": {
        "type": "object",
        "properties": {
            "op": {
                "type": "string",
                "description": "operator",
            },
            "input1": {
                "type": "number",
                "description": "input1",
            },
            "input2": {
                "type": "number",
                "description": "input2",
            },
        },
        "required": ["op", "input1", "input2"],
    },
}

ALL_TOOL_SPECS = [GET_PRODUCT_SPEC, CALCULATE_SPEC]

# ── エージェント ──────────────────────────────────────────────────────

def call_claude(messages, tools, model=None):
    return client.messages.create(
        model=model or MODEL,
        system=SYSTEM_PROMPT,
        max_tokens = 1024,
        tools=tools,
        messages=messages,
    )


def execute_tool(name, inputs):
    try:
        return str(TOOL_REGISTRY[name](**inputs))
    except Exception as e:
        return f"Error: {e}"


def run_agent(prompt, eval_mode=False, model=None):
    messages = [{"role": "user", "content": prompt}]
    total_input_tokens = 0
    total_output_tokens = 0

    while True:
        response = call_claude(messages, tools=ALL_TOOL_SPECS, model=model)
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        messages.append({"role": "assistant", "content": response.content})

        # Claudeがツールを要求した場合を除き、ループを抜ける。"end_turn" ではなく "tool_use" で
        # 判定することで、モデルが別の理由（例：max_tokens）で停止した場合に、400エラーが
        # 繰り返し発生することを防ぐ。そうしないと、空の tool_results メッセージを返送してしまう。
        # （サーバー側ツールを使う場合は、stop_reason == "pause_turn" も処理すること。）
        if response.stop_reason != "tool_use":
            break

        tool_calls = [block for block in response.content if isinstance(block, ToolUseBlock)]

        tool_results = []
        for tool_call in tool_calls:
            result = execute_tool(tool_call.name, tool_call.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_call.id,
                "content": result,
            })

        messages.append({"role": "user", "content": tool_results})

    if eval_mode:
        return {
            "messages": messages,
            "usage": {"input_tokens": total_input_tokens, "output_tokens": total_output_tokens},
        }

    return "\n".join(block.text for block in response.content if isinstance(block, TextBlock))


print("boutiqueエージェントの準備が完了しました。")

### 動作確認

下のセルを実行すると、エージェントとの対話セッションが自動的に開始されます。クエリを入力して、応答を確認してください。終了するには `quit` と入力します。（Run Allでノートブック全体を実行する場合は、事前にそのセルで `INTERACTIVE_CHAT = False` に設定するか、ここで停止した際に `quit` と入力してください。）

確認用のクエリ例：
- 「ジーンズはいくらですか？」（`How much do jeans cost?`）：単純な質問のため、正しく動作するはずです
- 「Tシャツの価格はいくらですか？」（`Price of a t-shirt?`）：ハイフンの扱いで誤りが生じるでしょうか
- 「靴はいくらですか？」（`How much for shoes?`）：カタログに「shoes」はありませんが、「sneakers」はあります
- 「シャツ3枚とベルト2本で、合計はいくらですか？」（`3 shirts and 2 belts, what's my total?`）：商品検索と計算の両方のツールが必要です
- 「ジャケットの20%オフはいくらですか？」（`What's 20% off a jacket?`）：パーセントの計算が必要です
- 「何を扱っていますか？」（`What do you sell?`）：エージェントは自らの機能を説明できるでしょうか

In [ ]:
ANTHROPIC_ORANGE = "#E07A5F"  # ブランドのアクセントカラー。ノートブックの白地に黒の出力の中で、boutiqueエージェントの出力が埋もれないようにする

def _display_chat_hint():
    """対話チャットの有効化方法を表示する。ノートブックではAnthropicオレンジ、スクリプト実行時は
    プレーンテキストで表示する。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("ノートブックのカーネル外のため、プレーンテキストのバナーを使用する")
        from IPython.display import display, HTML
        display(HTML(
            f'<div style="background:{ANTHROPIC_ORANGE};color:#fff;padding:10px 16px;'
            f'border-radius:8px;font-family:sans-serif;font-size:14px;">'
            f'🛍️ <b>The Boutique Agent</b> is ready — set '
            f'<code style="background:rgba(255,255,255,.3);padding:1px 5px;border-radius:4px;">INTERACTIVE_CHAT = True</code> '
            f'above and run this cell again to chat.</div>'
        ))
    except Exception:
        print("Boutique Agentの準備が完了しました。チャットを開始するには、上で INTERACTIVE_CHAT = True に設定し、このセルを再実行してください。")


INTERACTIVE_CHAT = True  # デフォルトではエージェントとのチャットを開始する。評価のセクションに直接進む場合は False に設定してRun Allを実行

if not INTERACTIVE_CHAT:
    _display_chat_hint()
else:
    print("Boutique Agentの応答結果")

# 下の input() プロンプトはVS Code自体のエディターUI（コマンドパレットと同様）であり、
# 色は常にVS Codeのテーマに従うため、ここからは変更できない。調整できるのはテキストのみのため、
# プロンプトの文言そのものでブランドを表現している。
while INTERACTIVE_CHAT:
    query = input("\n🛍️  Boutiqueショッピングエージェントが稼働中です。質問を入力してEnterキーを押してください。エージェントを停止するにはEscキーを押してください。")
    if not query.strip() or query.strip().lower() in ("quit", "exit", "q"):
        print("セッションを終了しました。")
        break
    print(f"\nBoutique: {run_agent(query)}")

---

## パート2：評価フレームワーク

評価は、主に3つのコンポーネントで構成されます。

```
Tasks ──> Runner ──> Graders ──> Results
```

- **タスク**：*何を*テストするかを定義します。シナリオ、期待される動作、採点方法を含みます。
- **ランナー**：実行全体を制御します。各シナリオのクエリをエージェントに送信し、トランスクリプトを収集して、採点者を適用します。
- **採点者**：エージェントが正しく動作した*かどうか*を確認し、スコアと理由を返します。

ランナーと採点者は、次の2つのセルで定義しています。両方を実行してください。仕組みを把握するためにコードに目を通しておくことをお勧めしますが、現時点では変更する必要はありません。

### 利用可能な採点者

| 採点者 | 確認内容 | チェックの形式 |
|--------|---------------|---------------|
| `response_contains` | 最終テキストに指定の文字列が含まれるか（大文字・小文字を区別しない） | `"49.99"` または `"jeans"` |
| `response_numeric` | 最終テキストに、許容誤差内の数値が含まれるか | `{"value": 49.99, "tolerance": 0.05}` |
| `tool_use` | エージェントが特定のツールを特定の引数で呼び出したか | `{"tool_name": "get_product", "arguments": {"product": "jeans"}}` |

各採点者は、2値のスコア（0 = 不合格、1 = 合格）と、その理由を返します。タスクが合格となるのは、**すべての採点者のすべてのチェック**に合格した場合のみです。

In [ ]:
# ── 採点者（このセルはそのまま実行）────────────────────────────────────

import re

def grade_response_contains(result, check, context=None):
    text = result["final_text"].lower()
    target = check.lower()
    if target in text:
        return {"score": 1.0, "reason": f"Found '{check}' in response"}
    return {"score": 0.0, "reason": f"'{check}' not found in response: {result['final_text'][:200]}"}


def grade_response_numeric(result, check, context=None):
    if isinstance(check, (int, float)):
        value, tolerance = float(check), 0.01
    else:
        value = float(check["value"])
        tolerance = float(check.get("tolerance", 0.01))

    numbers = re.findall(r"-?[\d,]+\.?\d*", result["final_text"])
    for num_str in numbers:
        try:
            num = float(num_str.replace(",", ""))
            if abs(num - value) <= tolerance:
                return {"score": 1.0, "reason": f"Found {num} (expected {value} +/- {tolerance})"}
        except ValueError:
            continue
    return {"score": 0.0, "reason": f"Expected {value} (+/- {tolerance}), found: {numbers[:10]}"}


def grade_tool_use(result, check, context=None):
    tool_name = check["tool_name"]
    expected_args = check.get("arguments", None)

    for call in result["tool_calls"]:
        if call["name"] != tool_name:
            continue
        if expected_args is None:
            return {"score": 1.0, "reason": f"Tool '{tool_name}' was called"}

        # 部分一致：指定されたキーのみを確認する
        actual_args = call.get("arguments", {})
        match = all(
            (isinstance(v, str) and isinstance(actual_args.get(k), str) and v.lower() == actual_args[k].lower())
            or actual_args.get(k) == v
            for k, v in expected_args.items()
        )
        if match:
            return {"score": 1.0, "reason": f"Tool '{tool_name}' called with matching args: {expected_args}"}

    actual = [{"name": c["name"], "args": c.get("arguments", {})} for c in result["tool_calls"]]
    if expected_args:
        return {"score": 0.0, "reason": f"'{tool_name}' not called with {expected_args}. Actual: {actual}"}
    return {"score": 0.0, "reason": f"'{tool_name}' never called. Actual: {[c['name'] for c in result['tool_calls']]}"}


GRADER_REGISTRY = {
    "response_contains": grade_response_contains,
    "response_numeric": grade_response_numeric,
    "tool_use": grade_tool_use,
}

print(f"Graders loaded: {list(GRADER_REGISTRY.keys())}")

In [ ]:
# ── 評価ランナー（このセルはそのまま実行）──────────────────────────────

import json, os, time, traceback
from concurrent.futures import ThreadPoolExecutor, as_completed


def parse_transcript(messages):
    """エージェントの生のトランスクリプトから final_text と tool_calls を抽出する。"""
    final_text, tool_calls = "", []
    for msg in messages:
        if msg["role"] != "assistant":
            continue
        for block in msg["content"]:
            if isinstance(block, TextBlock):
                final_text = block.text
            elif isinstance(block, ToolUseBlock):
                tool_calls.append({"name": block.name, "arguments": block.input, "id": block.id})
    # ツールの実行結果を、対応する呼び出しに紐付ける
    for msg in messages:
        if msg["role"] != "user" or not isinstance(msg["content"], list):
            continue
        for item in msg["content"]:
            if isinstance(item, dict) and item.get("type") == "tool_result":
                for call in tool_calls:
                    if call["id"] == item["tool_use_id"]:
                        call["result"] = item.get("content", "")
                        break
    return {"final_text": final_text, "tool_calls": tool_calls, "messages": messages}


def run_single_task(agent_fn, task, model=None):
    """タスクを1つ実行し、採点者を適用して、採点結果とメトリクスを含む結果を返す。"""
    start = time.time()
    try:
        raw = agent_fn(task["query"], eval_mode=True, model=model)
    except Exception:
        return {
            "task_id": task["id"], "task_description": task.get("description", ""),
            "query": task["query"], "category": task.get("category", ""),
            "error": traceback.format_exc(), "passed": False, "grades": [],
            "metrics": {"time": time.time() - start},
        }

    elapsed = time.time() - start
    result = parse_transcript(raw["messages"])
    usage = raw.get("usage", {})
    turns = sum(1 for m in raw["messages"] if m["role"] == "assistant")
    metrics = {
        "time": round(elapsed, 3), "tool_calls": len(result["tool_calls"]),
        "turns": turns, "input_tokens": usage.get("input_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
    }

    grades = []
    context = {"query": task["query"], "task_id": task["id"], "model": model}
    for grader in task.get("graders", []):
        grader_fn = GRADER_REGISTRY.get(grader["type"])
        if grader_fn is None:
            grades.append({"type": grader["type"], "check": None, "score": 0.0, "reason": f"Unknown grader: {grader['type']}"})
            continue
        for check in grader.get("checks", []):
            # 例外を送出する（または不正な値を返す）採点者は、そのタスクのみを失敗扱いにする。
            # このガードがない場合、run_eval 内の f.result() で例外が再送出され、
            # 実行全体が中断される。
            try:
                grade = grader_fn(result, check, context)
                grades.append({"type": grader["type"], "check": check, "score": grade["score"], "reason": grade["reason"]})
            except Exception as exc:
                grades.append({"type": grader["type"], "check": check, "score": 0.0,
                               "reason": f"grader error: {type(exc).__name__}: {exc}"})

    passed = all(g["score"] == 1.0 for g in grades) if grades else False

    return {
        "task_id": task["id"], "task_description": task.get("description", ""),
        "query": task["query"], "category": task.get("category", ""),
        "passed": passed, "grades": grades, "metrics": metrics,
        "final_text": result["final_text"],
        "transcript": [
            block.model_dump() if hasattr(block, "model_dump") else block
            for msg in raw["messages"]
            for block in (msg["content"] if isinstance(msg["content"], list) else [msg["content"]])
        ],
    }


def run_eval(agent_fn, tasks, model=None, num_runs=1, max_workers=5):
    """評価スイート全体を実行し、構造化された結果を返す。"""
    all_runs = []
    for _ in range(num_runs):
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(run_single_task, agent_fn, t, model): t for t in tasks}
            run_results = []
            for f in as_completed(futures):
                r = f.result()
                run_results.append(r)
                mark = "PASS" if r["passed"] else ("ERROR" if r.get("error") else "FAIL")
                print(f"  [{len(run_results)}/{len(tasks)}] {r['task_id']}: {mark}", flush=True)
        task_order = {t["id"]: i for i, t in enumerate(tasks)}
        run_results.sort(key=lambda r: task_order.get(r["task_id"], 999))
        all_runs.append(run_results)
    return {"runs": all_runs, "config": {"model": model, "num_runs": num_runs, "num_tasks": len(tasks)}}


def save_results(results, directory="eval_results"):
    """評価結果をJSONファイルに保存する。"""
    os.makedirs(directory, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    model_name = results["config"].get("model") or "default"
    model_short = model_name.split("-")[1] if "-" in str(model_name) else model_name
    filename = f"{directory}/eval_{model_short}_{timestamp}.json"
    with open(filename, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Results saved to {filename}")
    return filename


def print_summary(results):
    """評価結果を整形して表示する。"""
    config = results["config"]
    print(f"{'=' * 60}")
    print(f"EVAL RESULTS: {config['num_tasks']} tasks, {config['num_runs']} run(s)")
    if config.get("model"): print(f"Model: {config['model']}")
    print(f"{'=' * 60}\n")

    for run_idx, run in enumerate(results["runs"]):
        if config["num_runs"] > 1: print(f"--- Run {run_idx + 1} ---")
        passed = sum(1 for r in run if r["passed"])
        total = len(run)
        print(f"Overall: {passed}/{total} passed ({passed/total*100:.0f}%)\n")

        # カテゴリ別の内訳
        categories = {}
        for r in run:
            cat = r.get("category", "uncategorized")
            categories.setdefault(cat, {"passed": 0, "total": 0})
            categories[cat]["total"] += 1
            if r["passed"]: categories[cat]["passed"] += 1
        if len(categories) > 1:
            print("カテゴリ別：")
            for cat, c in sorted(categories.items()):
                print(f"  {cat}: {c['passed']}/{c['total']} ({c['passed']/c['total']*100:.0f}%)")
            print()

        # タスクごとの詳細
        print("Tasks:")
        for r in run:
            mark = "PASS" if r["passed"] else "FAIL"
            print(f"  [{mark}] {r['task_id']}: {r['task_description']}")
            for g in r.get("grades", []):
                print(f"    {'+' if g['score'] == 1.0 else '-'} {g['type']}: {g['reason'][:120]}")
            if r.get("error"): print(f"    Error: {r['error'][:200]}")

        # 集計メトリクス
        ok = [r for r in run if not r.get("error")]
        if ok:
            print(f"\nMetrics (avg): {sum(r['metrics']['time'] for r in ok)/len(ok):.2f}s, "
                  f"{sum(r['metrics']['tool_calls'] for r in ok)/len(ok):.1f} tool calls, "
                  f"{sum(r['metrics']['turns'] for r in ok)/len(ok):.1f} turns")
            print(f"Tokens: {sum(r['metrics']['input_tokens'] for r in ok):,} in, "
                  f"{sum(r['metrics']['output_tokens'] for r in ok):,} out")
        print()


def inspect_task(results, task_id, run_index=0):
    """特定のタスクの詳細な結果を、トランスクリプトを含めて表示する。"""
    run = results["runs"][run_index]
    r = next((r for r in run if r["task_id"] == task_id), None)
    if r is None:
        print(f"Task '{task_id}' not found"); return

    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {r['task_id']}: {r['task_description']}")
    print(f"Query: {r['query']}")
    print(f"Response: {r.get('final_text', 'N/A')}\n")
    if r.get("error"): print(f"ERROR:\n{r['error']}"); return

    print("Grades:")
    for g in r["grades"]:
        print(f"  {'+' if g['score'] == 1.0 else '-'} {g['type']}: {g['reason']}")
    print(f"\nMetrics: {r['metrics']}")

    print("\nTranscript:")
    for item in r.get("transcript", []):
        if isinstance(item, dict):
            t = item.get("type", "?")
            if t == "text": print(f"  [text] {item.get('text', '')[:300]}")
            elif t == "tool_use": print(f"  [tool_use] {item.get('name', '?')}({item.get('input', {})})")
            elif t == "tool_result": print(f"  [tool_result] {str(item.get('content', ''))[:200]}")
            else: print(f"  [{t}] {str(item)[:200]}")
        else: print(f"  {str(item)[:200]}")


print("評価フレームワークの準備が完了しました。")

### 設計上の留意点

上の採点者とランナーのコードには、次の設計判断が反映されています。

- **if/elifではなくレジストリのdict**：採点者は、通常のdictである `GRADER_REGISTRY` から種類名で参照します。新しい種類の採点者を追加する場合は、関数1つとdictのエントリ1つを追加するだけで済み、分岐処理を変更する必要はありません。多くの評価フレームワークで採用されているパターンです。

- **ランナーは出力（print）を行わない**：`run_eval()` はデータ構造を返し、`print_summary()` は別の関数として分けています。これにより、ランナーのテストと再利用が容易になります。ランナーを変更せずに、表示方法の差し替え、JSONへの保存、ダッシュボードへの連携が可能です。

- **エージェントを引数として受け取る**：ランナーはエージェントを直接importせず、`agent_fn` を引数として受け取ります。これにより、モックを使ったテスト、別のエージェントへの差し替え、計測用のラッパーの追加が容易になります。

- **トランスクリプトの解析はランナーが担う**：エージェントは生のメッセージを返し、ランナーが `final_text` や `tool_calls` などを抽出します。抽出ロジックの重複を避け、エージェントの実装を簡潔に保てます。エージェントは、自らがどのように評価されるかを知る必要がありません。

- **実行の失敗 ≠ タスクの失敗**：エージェントが例外を送出した場合（APIエラー、タイムアウト）は*実行の失敗*であり、`error` フィールドに記録され、採点は行われません。エージェントが誤った答えを返した場合は*タスクの失敗*であり、採点者が通常どおり動作して問題点を報告します。この区別は重要です。エラーはインフラの障害を、失敗はエージェントの誤答を意味します。

- **並列実行**：タスクは、I/OバウンドなAPI呼び出しの標準的な手法である `ThreadPoolExecutor` で並列に実行します。並列数を無制限にすると、レート制限に達します。エージェントが非同期の `AsyncAnthropic` クライアントを使用する場合は、`asyncio.gather()` に置き換えます。

---

## パート3：評価タスクの定義

### タスクのスキーマ

各タスクは、次のフィールドを持つPythonのdictです。

```python
{
    "id": "unique_task_id",             # Short, descriptive identifier
    "description": "What this tests",    # Human-readable description
    "query": "The user's question",      # What gets sent to the agent
    "category": "product_lookup",        # For grouping results
    "graders": [                          # List of grader declarations
        {
            "type": "response_contains",  # Which grader to use
            "checks": ["49.99"],          # What to check (one or more)
        },
    ],
}
```

### 作成例

ジーンズの価格を検索できるかを確認するタスクの例です（下のコードセルでは、クエリは日本語の「ジーンズはいくらですか？」としています）。

```python
{
    "id": "price_jeans",
    "description": "Direct price lookup for jeans",
    "query": "How much do jeans cost?",
    "category": "product_lookup",
    "graders": [
        {"type": "response_contains", "checks": ["49.99"]},
        {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
    ],
}
```

このタスクでは、2つの採点者を使用しています。
1. `response_contains`：最終的な回答に「49.99」が含まれているかを確認します
2. `tool_use`：エージェントが実際に `get_product` を `product="jeans"` で呼び出したか（価格をハルシネーションで生成していないか）を確認します

### `tool_use` 採点者の脆さ

ツール使用のチェックで検証すべきは、エージェントが回答の*根拠をツールに置いているか*（価格をハルシネーションで生成したり、暗算で済ませたりしていないか）であり、厳密な呼び出し順序を強制することではありません。想定外であっても妥当な経路で正しい答えに到達していれば、問題ありません。

`{"tool_name": "get_product"}`（引数なし）をチェックすると、エージェントがそのツールを使用したかどうかを確認できます。`{"tool_name": "get_product", "arguments": {"product": "jeans"}}` をチェックすると、引数の値まで確認できます。後者は、引数の値そのものがテスト対象となる場合（例：同義語の解決）に限って使用してください。

### ✏️ 実装課題

ジーンズのタスクは参考例として用意しています。次の5つのクエリ例について、タスクを作成してください。

1. 「Tシャツの価格はいくらですか？」（`"Price of a t-shirt?"`）：エージェントはハイフンを正しく扱えるでしょうか。`get_product` にはどのような引数を渡すべきでしょうか。
2. 「靴はいくらですか？」（`"How much for shoes?"`）：「shoes」はカタログにありません。本来どのような動作が望ましく、どのように採点しますか。
3. 「シャツ3枚とベルト2本で、合計はいくらですか？」（`"3 shirts and 2 belts, what's my total?"`）：複数の商品検索と計算が必要です。期待される合計金額はいくらでしょうか。
4. 「ジャケットの20%オフはいくらですか？」（`"What's 20% off a jacket?"`）：商品検索と計算の両方が必要です。どの数値をチェックすべきでしょうか。
5. 「何を扱っていますか？」（`"What do you sell?"`）：自由回答の質問です。現在の採点者で採点できるでしょうか。（ヒント：このタスクには、パート6の後で改めて取り組んでください。）

各クエリについて、次の点を検討してください。
- どのカテゴリに属するか
- どの採点者が適切か。どのようなチェックを設定するか
- 期待される正しい答えは何か
- `tool_use` のチェックは必要か。応答のチェックだけで十分か

### ✏️ 実装課題：評価タスクはこのセルに記述

各タスクのdictには、**id**、**query**（エージェントに送信するプロンプト）、合否を判定する**チェック**を含む**採点者**が必要です。作成例はそのまま残し、新しいタスクごとにコメントアウトされたテンプレートをコピーして、`# ✏️ ADD YOUR TASKS BELOW` マーカーの下に追加してください。

In [ ]:
# ✏️ 実装課題：評価タスクはこのリストに記述してください
# 各タスクに必要な項目：id、エージェントに送るクエリ（プロンプト）、チェックを含む採点者
tasks = [
    # ── 参考タスク（作成例）──────────────────────────────────────────
    {
        "id": "price_jeans",
        "description": "ジーンズの価格の直接検索",
        "query": "ジーンズはいくらですか？",
        "category": "product_lookup",
        "graders": [
            {"type": "response_contains", "checks": ["49.99"]},
            {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
        ],
    },

    # ── テンプレート：新しいタスクごとにこの骨組みをコピーしてください ──────
    # {
    #     "id": "short_unique_id",
    #     "description": "このタスクで検証する内容",
    #     "query": "エージェントに送信するユーザーメッセージ（実際の文面）",
    #     "category": "group_name_for_results",
    #     "graders": [
    #         {"type": "which_grader_to_use", "checks": ["チェックする内容"]},
    #     ],
    # },

    # ── 次のクエリのタスクを作成してください ──────────────────────────

    # 1. "Tシャツの価格はいくらですか？"（"Price of a t-shirt?"）

    # 2. "靴はいくらですか？"

    # 3. "シャツ3枚とベルト2本で、合計はいくらですか？"

    # 4. "ジャケットの20%オフはいくらですか？"

    # 5. "何を扱っていますか？"

    # ✏️ ADD YOUR TASKS BELOW

]

---

## パート4：評価の実行

評価を実行し、エージェントの性能を確認します。結果はJSONファイルとしても保存されるため、実行結果どうしを比較できます。

In [ ]:
results = run_eval(run_agent, tasks)
print_summary(results)
save_results(results)

### 結果の確認

サマリーからは、*何が*合格し、何が不合格だったかがわかります。*その理由*を把握するには、個々のタスクを確認します。

不合格のタスクについては、次の点を確認してください。
- エージェントは実際にどのように応答したか
- どのツールを、どの引数で呼び出したか
- どこで誤ったか：ツールの選択、引数の形式、最終的な回答のいずれか

合格したタスクについては、正しい理由で合格しているか、偶然の合格ではないかを確認してください。

あわせて、不合格という判定そのものが妥当かどうかも確認してください。

In [ ]:
# 確認したいタスクIDに置き換えてください
inspect_task(results, "price_jeans")

### ベースラインの確立

LLMの出力は非決定的です。最新のClaudeモデルではtemperatureなどのサンプリングパラメータを指定できません。また以前から、パラメータを固定しても同一の出力は保証されていませんでした。変更を加える前に評価を複数回実行し、信頼できるベースラインを取得してください。

In [ ]:
baseline = run_eval(run_agent, tasks, num_runs=5)
print_summary(baseline)

---

## パート5：エージェントの改善

評価結果から得た知見を活用して、エージェントを改善します。

変更できる箇所：
- **ツールの説明**：現在の仕様は意図的にあいまいにしています。Claudeがツールを正しく使用するには、どのような情報が必要でしょうか。
- **システムプロンプト**：`"You are a helpful assistant."` では、ショッピングアシスタントであるという文脈がClaudeにまったく伝わりません。
- **エラー処理**：商品が見つからない場合や、無効な演算子が使われた場合に、どのような動作になるでしょうか。
- **ツールの実装**：入力形式に対して、より柔軟に対応できないでしょうか。

変更後は、**上にスクロールしてエージェントのセルを再実行**してから評価を再実行し、変更の効果を確認してください。

---

## パート6：LLM-as-Judge採点者の追加

決定的な採点者では採点できないクエリもあります。例：
- *「何を扱っていますか？」*：カタログの説明の仕方には、妥当なものが何通りもあります。
- *「シャツ2枚とジャケット1着では、どちらがお得ですか？」*：数値だけでなく、推論を伴う回答が必要です。
- *「予算は100ドルです。何を購入すればよいですか？」*：妥当な提案であれば、いずれも正解です。

こうしたクエリには**LLM-as-Judge**、つまりClaude自身を使って応答を評価する採点者が必要です。

### インターフェースの仕様

採点者のインターフェースは、決定的な採点者と同じです。

```python
def grade_llm_judge(result, check, context=None):
    # check: a string describing what to evaluate, e.g.
    #   "Response lists available product categories"
    #   "Response correctly identifies the better deal with reasoning"
    #
    # context: includes the original query (context["query"]) and task metadata
    #
    # Returns: {"score": 0.0 or 1.0, "reason": "..."}
```

各チェックは、自然言語で記述した基準です。採点者はエージェントの応答と基準をClaudeに送信し、合否の2値判定を求めます。

### 実装のヒント

- **プロンプトの構成**：ジャッジ（判定役のモデル）には、元のクエリ、エージェントの応答（`result["final_text"]`）、基準（`check`）を渡し、基準を満たしているかの評価を求めます。
- **構造化された回答の強制**：1行目に `PASS` または `FAIL`、続けて理由を返すよう指示します。これにより、解析が容易になります。
- **1回の呼び出しで1つの基準**：各チェックは単一の基準です。1回のLLM呼び出しで複数の基準を評価させないでください。

### 実装

下の採点者を実装して登録し、その採点者を使うタスクを作成してください。

In [ ]:
# LLM-as-Judge採点者の実装
#
# 構造化出力（output_config.format）を使用するため、判定結果はenum付きの検証済みJSONとして返る。
# 自由記述の "PASS"/"FAIL" を文字列解析する必要はなく、
# サンプリングパラメータも使用しない（temperatureは最新モデルで廃止されており、
# 決定性はパラメータ調整ではなくスキーマによって担保される）。

JUDGE_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "verdict": {"type": "string", "enum": ["PASS", "FAIL"]},
            "reason": {"type": "string", "description": "理由を1文で記述。"},
        },
        "required": ["verdict", "reason"],
        "additionalProperties": False,
    },
}


def grade_llm_judge(result, check, context=None):
    # TODO: この採点者を実装する
    #
    # ステップ1：判定用プロンプトを作成する
    #   - 含める情報：context["query"]、result["final_text"]、チェックの基準
    #
    # ステップ2：構造化された判定形式を指定してClaudeを呼び出す
    #   - response = client.messages.create(model=FAST_MODEL, ...,
    #         output_config={"format": JUDGE_SCHEMA})
    #
    # ステップ3：判定結果を読み取る
    #   - data = json.loads(response.content[0].text)
    #   - 戻り値：{"score": 1.0 if data["verdict"] == "PASS" else 0.0, "reason": data["reason"]}
    #
    # 未実装の採点者が実行全体をクラッシュさせず、わかりやすく失敗するための仮の実装。
    # ご自身の実装に置き換えてください：
    return {"score": 0.0,
            "reason": "grade_llm_judge はまだ実装されていません。「LLM-as-Judge採点者の実装」セルで実装してください"}


# ランナーが使用できるように登録する
GRADER_REGISTRY["llm_judge"] = grade_llm_judge

In [ ]:
# LLM-as-Judge採点者を使うタスクの追加

llm_judge_tasks = [
    # {
    #     "id": "capabilities",
    #     "description": "エージェントが自らの機能を説明する",
    #     "query": "どのようなことに対応していますか？",
    #     "category": "capabilities",
    #     "graders": [
    #         {"type": "llm_judge", "checks": [
    #             "応答が、商品価格を検索できることに言及している",
    #             "応答が、計算を実行できることに言及している",
    #         ]},
    #     ],
    # },
]

# 両方のタスクセットで評価を実行する
# all_tasks = tasks + llm_judge_tasks
# results = run_eval(run_agent, all_tasks)
# print_summary(results)

---

## 発展課題

早く完了した方向けに、おおよそ難易度の低い順にアイデアを紹介します。

### 短時間で取り組める課題
- **ネガティブテストケース**：エージェントが適切に対応すべきクエリを追加してください。例：「ピアノはいくらですか？」（カタログ外）、「人生の意味とは？」（対象外の話題）。
- **タスクの検証**：評価の実行前に、すべてのタスクのdictに必須フィールドがあるかを確認する関数を作成してください。
- **Claudeによるタスク生成**：カタログとこれまでに作成したパターンをもとに、追加の評価タスクを生成するようClaude Codeに依頼してください。

### モデルの比較
- **Haiku vs Sonnet vs Opus**：同じ評価を複数のモデルで実行してください。より高性能なモデルは、不十分なツール仕様を補えるでしょうか。`run_eval(run_agent, tasks, model="claude-sonnet-5")` を使用して結果を比較してください。
- **コストと精度**：トークン使用量と、1回の実行あたりの推定コストを集計してください。Haikuがタスクの90%に合格する場合、Sonnetの追加コストに見合う価値はあるでしょうか。

### 採点者の高度化
- **効率の採点者**：正確性だけでなく効率も採点してください。エージェントは最小限のツール呼び出し回数で完了したでしょうか。
- **採点者の自動登録**：手動で管理している `GRADER_REGISTRY` dictを、関数を自動登録する `@grader` デコレーターに置き換えてください。

### 分析と可視化
- **結果の可視化**：カテゴリ別の合格率やモデル間の比較を、グラフ（matplotlib、plotly）で表示してください。
- **ベースラインとの比較**：保存した2つのJSON結果を並べて差分を取り、リグレッションを検出してください。
- **Pass@kとpass^k**：各タスクをk回実行してください。Pass@kは「少なくとも1回合格」、pass^kは「毎回合格」を意味します。不安定なタスクと、確実に失敗するタスクの区別に有効です。
- **統計的な頑健性**：複数回の実行における分散、信頼区間、統計的有意性を算出してください。

### ハーネスの改善
- **最大ターン数のガード**：ターン数の上限を設け、エージェントの無限ループを防いでください。
- **タスクごとのタイムアウト**：時間がかかりすぎるタスクを強制終了してください。
- **バックオフ付きリトライ**：一時的なAPIエラーに適切に対処してください。
- **キャッシュ**：クエリとエージェント設定のハッシュを用いて、変更のないタスクの再実行を省いてください。
- **進捗の表示**：評価の実行中に、リアルタイムでプログレスバーを表示してください。
- **パラメータスイープ**：プロンプト、エフォートレベル、モデルなどの全組み合わせ（直積）で実行してください。

---

## 本セッションの先へ

### 簡略化した点

本演習の評価では、一部の設計を意図的に簡略化しています。本番環境のシステムとの違いは次のとおりです。

- **エージェントの計測**：本演習では、エージェントが完全なトランスクリプトを返すように `eval_mode` フラグを追加しました。本来は、エージェント自体に手を加えないのが望ましい方法です。本番のフレームワークはエージェントをラップしてAPI呼び出しを捕捉し、エージェントのコードを変更せずにトランスクリプトを構築します。

- **オブザーバビリティ**：本演習では、完全なトランスクリプトをフラットなリストとして記録しています。本番のシステムではOpenTelemetryのスパンツリーを用いて、タイミング、入れ子構造、メタデータを含む構造化トレースを取得します。これにより、*何が*起きたかだけでなく、各ステップに*どれだけ時間を要したか*、呼び出しがどのように入れ子になっていたかまで把握できます。

- **環境の分離**：本演習のエージェントは、評価と同じプロセスで動作しています。コードを実行する評価（SWE-benchなど）では、3層のDockerパターンが標準です。(1) OSとランタイムを含むベースイメージ、(2) 依存関係をインストールした環境イメージ（ハッシュによるコンテンツアドレス方式で、タスク間で共有）、(3) タスクごとに特定のリポジトリをチェックアウトしたインスタンスイメージ。

- **タスクの形式**：本演習ではインラインのPython dictを使用しています。本番のハーネスでは通常、JSONLファイル（1行に1タスク）、外部ファイルを参照するYAML、またはデータベースによるタスクストアを使用します。

### 既存のフレームワーク

評価ハーネスを一から構築する必要はありません。次のフレームワークは、こうした機能を標準で提供しています。

- **Promptfoo**：使いやすいCLIとWeb UIを備えたオープンソースの評価フレームワーク。
- **Braintrust**：ロギング、トレーシング、比較ツールを備えたホスト型の評価プラットフォーム。
- **LangSmith**：LangChainエコシステムのトレーシング・評価プラットフォーム。
- **Harbor**：LLMの安全性に重点を置いたオープンソースの評価フレームワーク。

Anthropic社内では、**eval-toolbox**を使用して**eval-registry**に評価を追加しています。本セッションの次のステップとしては、作成した評価をこれらのフレームワークのいずれかに移行することが考えられます。